# BorakBot — QLoRA round 1 (Colab)

Fine-tunes `meta-llama/Llama-3.2-3B-Instruct` on the 506 committed training pairs
using LLaMA-Factory, 4-bit, on one free T4. Produces the adapter that Stage 3 of the
pipeline serves. Config and its reasoning: `training/qlora_config.yaml`.

**Runtime → Change runtime type → T4 GPU** before running. On CPU this will not finish.

Run the cells **in order**.

    code        -> GitHub          (re-cloned every session)
    base model  -> Hugging Face    (~6 GB, re-downloaded after each disconnect)
    adapter     -> Google Drive    (checkpointed every 25 steps), then HF Hub

Roughly **45-75 minutes** for ~158 steps. Colab disconnects on idle; if that happens,
re-run Cells 1-3 and then **Cell 5 (resume)** instead of Cell 4. Checkpoints live on
Drive, so at most 25 steps of work is lost.

## Cell 1 — Setup

Mounts Drive, installs LLaMA-Factory, clones the repo. Four to six minutes, most of it pip.

Same three hardening measures as `colab_bakeoff.ipynb` Cell 1, each of which exists
because it already went wrong once: `os.chdir('/content')` before deleting the clone,
`shutil.rmtree` on a path constant rather than `!rm -rf` on a path containing spaces,
and a stale-clone assertion that fails here instead of three cells later.

**LLaMA-Factory is pinned to `v0.9.5`** — the current release as of August 2026.
Pinning a tag rather than tracking `main` keeps a graded run reproducible; on `main`
an upstream change can break a run mid-week with nothing to diff against.

Two things about this install that are not obvious:

- **v0.9.5 declares no extras.** `pip install -e ".[torch,bitsandbytes]"` does not
  fail, it just warns and installs neither group. Everything ML is a *required*
  dependency in its `pyproject.toml`, so the plain editable install is correct — but
  **bitsandbytes is not a dependency at all** and has to be installed separately.
  `qlora_config.yaml` sets `quantization_method: bnb`, so without it the run dies at
  model load.
- **This will change the `transformers` version** already in the runtime.
  LLaMA-Factory v0.9.5 constrains `transformers<=5.6.0`, which is *not* the
  `5.15.0` that `requirements.txt` pins. That is fine and expected:
  `requirements.txt` describes the local CPU environment that runs the Streamlit app,
  not this training runtime. The two never share a process.

If pip reports that a restart is needed: **Runtime → Restart session**, then re-run
this cell. Drive stays mounted; the "already mounted" notice on a re-run is
informational, not an error.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, subprocess

REPO = pathlib.Path('/content/NLP-BorakBot')
URL  = 'https://github.com/yongvay/NLP-BorakBot.git'
LF   = pathlib.Path('/content/LLaMA-Factory')

# Step out of REPO before removing it. Deleting the directory the process is standing
# in leaves an invalid cwd and everything after fails on getcwd.
os.chdir('/content')

# Pinned, not tracking main -- see the note above.
if not LF.exists():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v0.9.5',
                    'https://github.com/hiyouga/LLaMA-Factory.git', str(LF)], check=True)

# No extras: v0.9.5 declares none, and asking for them only prints a warning.
!pip install -q -e /content/LLaMA-Factory
# Not a LLaMA-Factory dependency, but qlora_config.yaml sets quantization_method: bnb.
!pip install -q bitsandbytes

# No shell, so no word-splitting on the path and nothing outside REPO can be hit.
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '-q', URL, str(REPO)], check=True, cwd='/content')

%cd /content/NLP-BorakBot
!git log --oneline -1

# This clone is GitHub's copy, not the laptop's working tree. If the config still
# names the superseded Qwen placeholder, the local edit was never pushed -- and the
# run would train the wrong model under the wrong template without erroring.
cfg = (REPO / 'training' / 'qlora_config.yaml').read_text(encoding='utf-8')
assert 'Llama-3.2-3B-Instruct' in cfg, (
    'STALE CLONE: the pushed qlora_config.yaml is not the Llama one. '
    'Commit and push training/qlora_config.yaml, then re-run this cell.')
print('clone ok')

import bitsandbytes, torch
print('bitsandbytes:', bitsandbytes.__version__)
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Cell 2 — Hugging Face token

**Required.** `meta-llama/Llama-3.2-3B-Instruct` is gated. Access has been granted on
the account, but the runtime still has to authenticate as that account.

Add it through the **key icon in the left sidebar** (Colab Secrets): name `HF_TOKEN`,
paste the value, switch on *Notebook access*. Never paste the token into a cell —
notebook source gets committed and output gets shared.

Unlike the bake-off there is no ungated fallback candidate here, so this cell fails
loudly rather than warning. A missing token would otherwise surface as a model-load
error after the dataset has already been built.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))
print('HF login ok')

## Cell 3 — Build the dataset, then check three things

`training/data/` is **gitignored**, so it is not in the fresh clone and has to be
rebuilt here. `to_llamafactory.py` reads the committed splits and stamps the system
prompt onto every example.

The three assertions below each guard a failure that does **not** raise on its own:

1. **Template.** A wrong chat template does not error — it trains on mis-delimited
   text and produces a model that rambles past its stop token. This is exactly what
   invalidated MaLLaM's round-1 bake-off run.
2. **Example count.** Catches a partial or stale split before the run rather than after.
3. **System prompt present.** The weights teach refusal and the prompt states the
   boundary; that only holds if the model was trained with the prompt in context.
   Train without it and serve with it, and the prompt reads as noise.

In [ ]:
!python training/to_llamafactory.py

from llamafactory.data.template import TEMPLATES
assert 'llama3' in TEMPLATES, 'template llama3 missing from this LLaMA-Factory build'
print('template llama3: ok')

import json, pathlib
info  = json.loads(pathlib.Path('training/data/dataset_info.json').read_text(encoding='utf-8'))
train = json.loads(pathlib.Path('training/data/borakbot_train.json').read_text(encoding='utf-8'))
assert set(info) == {'borakbot_train', 'borakbot_val'}, list(info)
assert len(train) == 506, f'expected 506 train examples, got {len(train)}'
assert train[0]['system'], 'system prompt missing -- train and serve would disagree'
print(f'dataset ok: {len(train)} train examples, system prompt stamped on each')

## Cell 4 — Train

`dataset_dir` in the YAML is a **relative** path, so this has to run from the repo root.

Roughly 45-75 minutes. What to watch in the log:

- **`loss`** should fall steadily. Flat from the start usually means the template or
  the data path is wrong, not that the learning rate is too low.
- **`eval_loss`**, printed every 25 steps. `qlora_config.yaml:79-83` sets 5 epochs
  deliberately because the goal is memorisation, but if eval loss turns up while train
  loss keeps falling, the useful checkpoint is the one before the turn — not the last
  one. Note the step number; you can point `--adapter` at
  `<output_dir>/checkpoint-<n>` instead of at the final directory.

Either way, **keep the loss curve**. It is the honest evidence for how much of the
corpus the model actually absorbed, and `plot_loss: true` writes it to the output
directory on Drive.

In [ ]:
%cd /content/NLP-BorakBot
!llamafactory-cli train training/qlora_config.yaml

## Cell 5 — Resume after a disconnect

**Run this instead of Cell 4** if Colab dropped the session mid-run.

The YAML sets `overwrite_output_dir: true`, which is right for a fresh run and wrong
for a resumed one — on re-run it starts from scratch and discards the Drive
checkpoints that `save_steps: 25` exists to create. Overriding it on the command line
keeps the YAML honest about the default (fresh) while making resume explicit:
LLaMA-Factory then picks up the newest `checkpoint-*` in `output_dir`.

Cells 1-3 still have to be re-run first — `/content` is wiped on disconnect.

In [ ]:
%cd /content/NLP-BorakBot
!llamafactory-cli train training/qlora_config.yaml --overwrite_output_dir False

## Cell 6 — Smoke test the adapter

The same 20 committed probes the bake-off used, now with the adapter attached. About
five minutes.

This is deliberately the **probe set, not `--all`** — it is a does-it-work check, and
reusing the committed probe set makes it directly comparable to the un-fine-tuned
`eval/results/llama.json` from the bake-off. The real evaluation over the full 63-item
test split is Step 5, not this notebook.

Two things to read in the output:

- Does it speak rojak now, rather than the base model's formal register?
- Does it still decline the out-of-knowledge probes? The base scored **3/3**. If
  fine-tuning taught it the register but cost it the refusals, that is the headline
  finding and it belongs in Part B.

In [ ]:
ADAPTER = '/content/drive/MyDrive/RDS3S1/NLP/borakbot-qlora-r1'

!python eval/generate.py \
    --model meta-llama/Llama-3.2-3B-Instruct \
    --adapter {ADAPTER} \
    --split test --4bit --tag tuned_smoke

!python eval/refusal_report.py --runs llama,tuned_smoke --show-misses

## Cell 7 — Push the adapter to the Hub

Weights never go into git (`CLAUDE.md`, working agreements). The rank-16 adapter is
~100 MB; the `checkpoint-*` directories are the every-25-steps safety copies and are
excluded — they are on Drive already and do not need a second home.

Change `HF_USER` to your Hugging Face username first.

In [ ]:
from huggingface_hub import HfApi

HF_USER = 'yongvay'
REPO_ID = f'{HF_USER}/borakbot-qlora-r1'

api = HfApi()
api.create_repo(REPO_ID, repo_type='model', private=True, exist_ok=True)
api.upload_folder(
    folder_path=ADAPTER,
    repo_id=REPO_ID,
    ignore_patterns=['checkpoint-*/**'],
)
print(f'https://huggingface.co/{REPO_ID}')

## Cell 8 — What happens next

1. Copy the loss curve from the output directory on Drive into `docs/` and commit it.
   Record the final train and eval loss in `docs/DESIGN.md`.
2. Commit `eval/results/tuned_smoke.json` — the smoke-test generations are evidence.
3. **Step 5, the graded comparison**: fine-tuned vs un-fine-tuned over the full test
   split, which is a different run from the smoke test above.

       python eval/generate.py --model meta-llama/Llama-3.2-3B-Instruct \
           --split test --all --4bit --tag base
       python eval/generate.py --model meta-llama/Llama-3.2-3B-Instruct \
           --adapter <hf-repo> --split test --all --4bit --tag tuned

   Then perplexity, BLEU, ROUGE-L, BERTScore, plus the human Likert pass. The scorer
   for this does not exist yet — `eval/generate.py`'s docstring calls it `eval/score.py`.
4. The prompt ablation, also named in that docstring: `--no-system-prompt
   --refusals-only`. It separates the two hallucination safeguards and shows how much
   of the refusal behaviour lives in the weights rather than in the prompt.